## Улучшение качества через TF-IDF + Logistic Regression

### 📋 Общее описание проекта

Этот блокнот реализует улучшенную модель анализа тональности текста с использованием TF-IDF векторизации и логистической регрессии. Вот основные этапы работы:

- **Загрузка и предобработка данных**: Используется датасет Sentiment140 с твитами, где целевая переменная кодируется как 0 (негатив) и 1 (позитив)
- **Очистка текста**: Реализована функция `clean_text()` для предобработки текста:
  - Приведение к нижнему регистру
  - Удаление URL-ссылок
  - Удаление упоминаний пользователей (@username)
  - Удаление всех символов кроме букв и пробелов
  - Нормализация пробелов
- **Векторизация текста**: Применяется TfidfVectorizer для преобразования очищенного текста в числовые признаки с параметрами:
  - Размер словаря: 300,000 самых важных слов
  - Подлогарифмическое масштабирование частот (sublinear_tf=True)
  - N-граммы: от 1 до 2 слов
  - Минимальная частота слова: 3 раза
- **Обучение модели**: Используется логистическая регрессия с оптимизацией:
  - Максимальное количество итераций: 1000
  - Оптимизатор: saga
  - Параметр регуляризации: C=2.0
  - Параллельная обработка: n_jobs=-1

### 🎯 Метод предсказания класса

**Комбинированный подход: TF-IDF + Logistic Regression**

Основные преимущества метода:

- **Повышенная точность**: TF-IDF учитывает не только частоту слов, но и их важность в документе и коллекции
- **Учет контекста**: Логистическая регрессия может улавливать сложные зависимости между словами
- **Регуляризация**: Параметр C=2.0 предотвращает переобучение
- **Масштабируемость**: Pipeline позволяет легко обновлять векторизацию и модель
- **Интерпретируемость**: Коэффициенты логистической регрессии показывают влияние каждого слова на предсказание

### 📊 Результаты модели

| Метрика | Значение |
|---------|----------|
| **Accuracy** | 0.8207 (82.07%) |
| **Precision (негатив)** | 0.83 |
| **Recall (негатив)** | 0.81 |
| **F1-score (негатив)** | 0.82 |
| **Precision (позитив)** | 0.81 |
| **Recall (позитив)** | 0.83 |
| **F1-score (позитив)** | 0.82 |

### 🔍 Сравнение с предыдущей моделью

| Модель | Accuracy | Время обучения |
|--------|----------|----------------|
| Naive Bayes + CountVectorizer | 79.92% | 31.16 сек |
| TF-IDF + Logistic Regression | **82.07%** | 48.24 сек |

### 🚀 Алгоритм работы

1. **Очистка текста**: Текст проходит этап предобработки для удаления шума
2. **TF-IDF векторизация**: Каждое слово получает вес, основанный на:
   - Частоте в документе
   - Редкости в коллекции документов
3. **Логистическая регрессия**: Модель изучает веса для каждого TF-IDF признака
4. **Предсказание**: Для нового текста вычисляется вероятность принадлежности к классам
5. **Принятие решения**: Вероятность сравнивается с порогом 0.5

Модель показывает улучшение точности на 2.15% по сравнению с базовым подходом, что делает ее более эффективной для анализа тональности текста.

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score

# Загружаем датасет
feature_names = ["target", "id", "date", "flag", "user", "text"]
df = pd.read_csv(
    "./data/Sentiment140.csv", sep=",", encoding="latin-1", names=feature_names
)

In [ ]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+", "", text)  # убрать ссылки
    text = re.sub(r"@\w+", "", text)     # убрать @username
    text = re.sub(r"[^a-z\s]", "", text) # убрать всё кроме букв
    text = re.sub(r"\s+", " ", text).strip()  # лишние пробелы
    return text

df['clean_text'] = df['text'].apply(clean_text)
display(df[['text', 'clean_text']].sample(5))

,text,clean_text
1343688,I've got 7 hours to get this essay done and ha...,ive got hours to get this essay done and hande...
1114179,keeps adding more students to the roster! I l...,keeps adding more students to the roster i lov...
1160332,@michmwong heyyy! I'm one happy camper at Cisc...,heyyy im one happy camper at cisco changing th...
1594460,Goodmorning twitter,goodmorning twitter
25465,@CarMaKi you missed the zombie walk...always ...,you missed the zombie walkalways next year tho...


## Модель TF-IDF + Logistic Regression

In [ ]:
start = time.time()

# Кодирует целевую переменную: 0 → 0 негатив, 4 → 1 позитив
df["target"] = df["target"].map({0: 0, 4: 1})

# Сохраняет очищенный текст
df["text_clean"] = df["text"].apply(clean_text)

# Разделение на трейн и тест
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["target"], stratify=df["target"], test_size=0.3, random_state=1
)

model = make_pipeline(
    TfidfVectorizer(
        max_features=300_000,  # макс. кол-во слов
        sublinear_tf=True,
        # Вариант 1
        analyzer="word",
        ngram_range=(1, 2),
        min_df=3,
        # Вариант 2
        # analyzer="char_wb", # разбить текст на подслова
        # ngram_range=(3, 5), # слова (от 3 - 5 символов)
        # min_df=5,
    ),
    LogisticRegression(max_iter=1000, solver="saga", C=2.0, n_jobs=-1),
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Время выполнения: {time.time() - start:.2f} сек.")

              precision    recall  f1-score   support

           0       0.83      0.81      0.82    240000
           1       0.81      0.83      0.82    240000

    accuracy                           0.82    480000
   macro avg       0.82      0.82      0.82    480000
weighted avg       0.82      0.82      0.82    480000

Accuracy: 0.8207
Время выполнения: 48.24 сек.


**Результат обучения модели**

Accuracy: 0.8207

---

In [4]:
import pandas as pd
import re

# Загружаем датасет
feature_names = ["target", "id", "date", "flag", "user", "text"]
df = pd.read_csv(
    "./data/Sentiment140.csv", sep=",", encoding="latin-1", names=feature_names
)

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+", "", text)  # убрать ссылки
    text = re.sub(r"@\w+", "", text)     # убрать @username
    text = re.sub(r"[^a-z\s]", "", text) # убрать всё кроме букв
    text = re.sub(r"\s+", " ", text).strip()  # лишние пробелы
    return text

# Кодирует целевую переменную: 0 → 0 негатив, 4 → 1 позитив
if 4 in df["target"].unique():
    df["target"] = df["target"].map({0: 0, 4: 1})

# Сохраняет очищенный текст
df["text_clean"] = df["text"].apply(clean_text)

print(df.shape)
display(df.sample(5))

(1600000, 7)


,target,id,date,flag,user,text,text_clean
1472636,1,2065413175,Sun Jun 07 08:32:39 PDT 2009,NO_QUERY,xoshayzers,"dude, now im dying to see Step up 3d even more...",dude now im dying to see step up d even more h...
100588,0,1794041457,Thu May 14 04:47:17 PDT 2009,NO_QUERY,princess_2895,wishes that she could go to the Mall of Asia h...,wishes that she could go to the mall of asia h...
1319925,1,2014529044,Wed Jun 03 01:50:19 PDT 2009,NO_QUERY,santanachamp,"@uponadream I watch True Blood, too! So does @...",i watch true blood too so does
1529229,1,2177521636,Mon Jun 15 06:15:10 PDT 2009,NO_QUERY,rhiannabananax,@Samanthaa_Sull good,good
1212447,1,1989249888,Mon Jun 01 00:52:05 PDT 2009,NO_QUERY,Glebe2037,http://bit.ly/PCDWg &lt;- for @happysinger an...,lt for and his type


In [6]:
import time
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

start = time.time()

# Разделение на трейн и тест
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["target"], stratify=df["target"], test_size=0.3, random_state=1
)

pipeline = Pipeline(
    [
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("logreg", LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)),
    ]
)

param_distributions = [
    {
        "tfidf__analyzer": ["word"],
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "tfidf__min_df": [2, 3, 5],
        "tfidf__max_features": [50_000, 100_000, 200_000, 300_000],
        "logreg__C": [0.3, 0.5, 1.0, 2.0, 3.0, 5.0],
        "logreg__penalty": ["l2"],
    },
    {
        "tfidf__analyzer": ["char_wb"],
        "tfidf__ngram_range": [(3, 5), (4, 6), (5, 7)],
        "tfidf__min_df": [3, 5, 7],
        "tfidf__max_features": [50_000, 100_000, 200_000],
        "logreg__C": [0.5, 1.0, 2.0, 3.0],
        "logreg__penalty": ["l2"],
    },
]

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

search.fit(X_train, y_train)

print("Лучшие параметры:")
print(search.best_params_)

print(f"\nЛучшая accuracy на кросс-валидации: {search.best_score_:.4f}")

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nРезультат на тестовой выборке:")
print(classification_report(y_test, y_pred))
print(f"Accuracy test: {accuracy_score(y_test, y_pred):.4f}")

print(f"Время выполнения: {time.time() - start:.2f} сек.")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END logreg__C=0.3, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=100000, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  22.9s
[CV] END logreg__C=0.3, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=100000, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  23.7s
[CV] END logreg__C=0.3, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=100000, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  23.6s
[CV] END logreg__C=3.0, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=200000, tfidf__min_df=3, tfidf__ngram_range=(1, 2); total time= 1.0min
[CV] END logreg__C=3.0, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=200000, tfidf__min_df=3, tfidf__ngram_range=(1, 2); total time= 1.2min
[CV] END logreg__C=3.0, logreg__penalty=l2, tfidf__analyzer=word, tfidf__max_features=200000, tfidf__min_df=3, tfidf__ngram_range=(1, 2); total time= 1.2m

KeyboardInterrupt: 